> 在一些任务上，GRPO 在数学上就退化成了"带正负样本的加权 SFT"

- https://arxiv.org/pdf/2505.13697：RL in Name Only? Analyzing the Structural Assumptions in RL post-training for LLMs 
    - 在 LLM 常用的 MDP 形式化下（state = token 拼接，terminal reward 均分到整条轨迹），GRPO 目标会塌缩成一个退化的加权交叉熵，等价于 outcome-driven SFT with positive and negative samples。他们据此提出 Filtered Iterative SFT，在 GSM8K/Countdown 上用 Qwen2.5 / Llama-3.2 能匹配 GRPO 的效果。
    - 输出短、reward 就是标签本身、group 内轨迹多样性低 —— 这正是退化条件最成立的场景。所以 RL 花了几倍算力，做的事情和 SFT 差别不大。
    - SFT 在数千样本量级时，接 RL 通常有 5–15pp 增益；但 SFT 规模进到几万条以后，SFT 和 SFT→RL 的 gap 会收窄。而对于定义明确的任务（分类、抽取、翻译），SFT alone 往往就已经接近上限。

### Gradient starvation：可能大部分 batch 根本没有梯度

- 如果 reward 是 exact-match，group 内很容易全对或全错，此时 group-mean centering 后 advantage 恒为 0。
- Gradient Starvation in Binary-Reward GRPO 给出退化概率 $D(p,G)=p^G+(1-p)^G$，
    - 并报告 Qwen3.5-9B 在 GSM8K、G=4 时退化率达 0.69 —— 近七成的 group 白算。
    - 诊断指标
        - 统计 RL 训练中 group 内 reward std == 0 的 batch 占比。>40% 说明算力大部分在空转。